# Lab 1: Vector Addition

---

## Overview

This lab introduces the fundamental programming model for AMD GPUs using HIP. You will learn how software abstractions (grids, blocks, threads) map to the underlying hardware (Compute Units, SIMDs, Wavefronts).

## Learning Objectives

By the end of this lab, you will be able to:

1. Describe the hierarchical thread organization in GPU programming
2. Explain how threads map to AMD GPU hardware components
3. Calculate global thread indices for kernel execution
4. Analyze the impact of block size on thread utilization and occupancy

## 1. The GPU Execution Model

GPU programming follows a **Single Instruction, Multiple Thread (SIMT)** model. A single kernel function is executed by thousands of threads simultaneously, each operating on different data.

### Hierarchical Thread Organization

Threads are organized into a three-level hierarchy:

```
Grid (Kernel Launch)
 |
 +-- Block 0
 |    +-- Thread 0, Thread 1, ..., Thread (blockDim-1)
 +-- Block 1
 |    +-- Thread 0, Thread 1, ..., Thread (blockDim-1)
 +-- ...
 +-- Block (gridDim-1)
      +-- Thread 0, Thread 1, ..., Thread (blockDim-1)
```

### Key Terminology

| Term | Definition |
|:-----|:-----------|
| **Kernel** | A function that executes on the GPU, launched from the host (CPU) |
| **Grid** | The complete set of threads executing a kernel |
| **Block** | A group of threads that can cooperate via shared memory and synchronization |
| **Thread** | The smallest unit of execution; each thread has a unique ID |
| **gridDim** | Number of blocks in the grid |
| **blockDim** | Number of threads per block |
| **blockIdx** | Index of the current block within the grid |
| **threadIdx** | Index of the current thread within its block |

## 2. AMD GPU Hardware Architecture

Understanding the hardware is essential for writing efficient GPU code. The programming model abstractions map directly to physical hardware components.

### Hardware Components

| Software Abstraction | AMD Hardware | Role |
|:---------------------|:-------------|:-----|
| Grid | GPU Device | Entire kernel workload |
| Block | Compute Unit (CU) | Independent execution unit |
| Warp (Wavefront) | SIMD Unit | 64 threads executing in lockstep |
| Thread | Vector Lane | Single execution context |

### Compute Unit (CU) Architecture

Each AMD GCN/CDNA Compute Unit contains:

| Component | Description |
|:----------|:------------|
| **4 SIMD Units** | Each has a 16-wide Vector ALU |
| **Vector Register File** | Private registers for each thread |
| **Local Data Share (LDS)** | 64 KB shared memory per CU |
| **L1 Vector Cache** | 16 KB per CU |
| **Scalar Unit** | Handles uniform operations across a warp |

### Wavefront (Warp) Execution

On AMD GPUs, a **wavefront** consists of **64 threads** that execute the same instruction simultaneously:

- The 64-thread wavefront is issued to a SIMD over 4 cycles (16 lanes x 4 cycles)
- All threads in a wavefront must execute the same instruction (SIMT)
- If threads diverge (take different branches), both paths are executed serially

**Important**: Block sizes should be multiples of 64 to avoid wasting execution resources.

## 3. Thread Indexing

Each thread must determine which data element it should process. This is done by computing a **global thread index**.

### 1D Indexing Formula

For a 1D grid with 1D blocks:

$$
\text{globalIdx} = \text{blockIdx.x} \times \text{blockDim.x} + \text{threadIdx.x}
$$

### Visual Example

Consider a grid with 3 blocks, each containing 4 threads:

```
Block 0              Block 1              Block 2
+---+---+---+---+    +---+---+---+---+    +---+---+---+---+
| 0 | 1 | 2 | 3 |    | 4 | 5 | 6 | 7 |    | 8 | 9 |10 |11 |  <- Global Index
+---+---+---+---+    +---+---+---+---+    +---+---+---+---+
  0   1   2   3        0   1   2   3        0   1   2   3    <- threadIdx.x
  blockIdx.x = 0       blockIdx.x = 1       blockIdx.x = 2
```

**Example Calculation** (Thread 5):
- `blockIdx.x = 1`, `blockDim.x = 4`, `threadIdx.x = 1`
- `globalIdx = 1 * 4 + 1 = 5`

### Exercise: Thread Index Calculation

Fill in the global index for each thread:

| blockIdx.x | blockDim.x | threadIdx.x | Global Index |
|:-----------|:-----------|:------------|:-------------|
| 0 | 256 | 100 | ______ |
| 2 | 128 | 50 | ______ |
| 5 | 64 | 0 | ______ |

## 4. Exploring Your GPU

Before writing kernels, examine the hardware specifications of your GPU.

In [ ]:
%%bash
# Query GPU device properties
hipinfo 2>/dev/null || rocminfo | grep -A 20 "Agent 2"

### Exercise 1: GPU Specifications

Based on the output above, record your GPU's specifications:

| Property | Your GPU Value | Significance |
|:---------|:---------------|:-------------|
| Device Name | ______ | GPU model identifier |
| Compute Units | ______ | Number of parallel execution units |
| Wavefront Size | ______ | Threads per warp (typically 64 on AMD) |
| Max Threads per Block | ______ | Upper limit for blockDim |
| Max Shared Memory per Block | ______ | LDS available per block |

**Question**: How many total threads can your GPU execute simultaneously if all CUs are fully utilized?

Your calculation: ______

## 5. Your First HIP Kernel

We will implement a vector addition kernel: $C[i] = A[i] + B[i]$

This simple kernel demonstrates the basic structure of GPU programming:
1. Each thread processes one element
2. Threads compute their global index
3. Boundary checking prevents out-of-bounds access

### Kernel Structure

```cpp
__global__ void vector_add(const float* A, const float* B, float* C, int N) {
    // Step 1: Calculate global thread index
    int idx = _______________________________;  // Fill in
    
    // Step 2: Boundary check (important when N is not divisible by block size)
    if (______) {  // Fill in the condition
        // Step 3: Perform computation
        C[idx] = _______________________________;  // Fill in
    }
}
```

### Exercise 2: Implement the Kernel

Complete the kernel in `vector_add_kernel.hip` based on the structure above.

In [ ]:
# View the kernel file
with open('vector_add_kernel.hip', 'r') as f:
    print(f.read())

## 6. Compile the Program

In [ ]:
%%bash
# Compile with hipcc
hipcc -O3 main_lab.cpp vector_add_kernel.hip -o vector_add_lab
echo "Compilation successful."

## 7. Launch Configuration

When launching a kernel, you must specify the **grid** and **block** dimensions.

### Kernel Launch Syntax

```cpp
hipLaunchKernelGGL(kernel_name, 
                   dim3(grid_size),    // Number of blocks
                   dim3(block_size),   // Threads per block
                   shared_mem_size,    // Dynamic shared memory (bytes)
                   stream,             // HIP stream (0 for default)
                   arg1, arg2, ...);   // Kernel arguments
```

### Calculating Grid Size

To ensure all N elements are processed:

$$
\text{grid\_size} = \left\lceil \frac{N}{\text{block\_size}} \right\rceil = \frac{N + \text{block\_size} - 1}{\text{block\_size}}
$$

### Thread Utilization

When N is not divisible by block_size, some threads in the last block are idle:

$$
\text{Utilization} = \frac{N}{\text{grid\_size} \times \text{block\_size}} \times 100\%
$$

### Exercise 3: Launch Configuration Calculation

Given `N = 1000` elements, complete the table:

| Block Size | Grid Size Formula | Grid Size | Total Threads | Idle Threads |
|:-----------|:------------------|:----------|:--------------|:-------------|
| 64 | ceil(1000/64) | ______ | ______ | ______ |
| 128 | ceil(1000/128) | ______ | ______ | ______ |
| 256 | ceil(1000/256) | ______ | ______ | ______ |
| 512 | ceil(1000/512) | ______ | ______ | ______ |

**Question**: Which block size gives the best thread utilization? Why might this not always be the best choice?

Your answer: ______

In [ ]:
# Verify your calculations
import math

N = 1000
block_sizes = [64, 128, 256, 512]

print(f"{'Block Size':>10} {'Grid Size':>10} {'Total Threads':>14} {'Idle Threads':>13} {'Utilization':>12}")
print("-" * 62)

for bs in block_sizes:
    grid_size = math.ceil(N / bs)
    total_threads = grid_size * bs
    idle_threads = total_threads - N
    utilization = (N / total_threads) * 100
    print(f"{bs:>10} {grid_size:>10} {total_threads:>14} {idle_threads:>13} {utilization:>11.2f}%")

## 8. Experiments: Block Size and Performance

Run the kernel with different block sizes to observe how it affects execution time.

In [ ]:
%%bash
# Experiment with block size 64
echo "=== Block Size: 64 ==="
./vector_add_lab 1048576 64

In [ ]:
%%bash
# Experiment with block size 128
echo "=== Block Size: 128 ==="
./vector_add_lab 1048576 128

In [ ]:
%%bash
# Experiment with block size 256
echo "=== Block Size: 256 ==="
./vector_add_lab 1048576 256

In [ ]:
%%bash
# Experiment with block size 512
echo "=== Block Size: 512 ==="
./vector_add_lab 1048576 512

In [ ]:
%%bash
# Experiment with block size 1024
echo "=== Block Size: 1024 ==="
./vector_add_lab 1048576 1024

### Exercise 4: Record Experimental Results

Fill in the execution times from your experiments:

| Block Size | Execution Time (ms) |
|:-----------|:--------------------|
| 64 | ______ |
| 128 | ______ |
| 256 | ______ |
| 512 | ______ |
| 1024 | ______ |

## 9. Wavefront Alignment

On AMD GPUs, the wavefront size is 64 threads. Block sizes that are not multiples of 64 result in partially filled wavefronts.

### Wavefront Occupancy by Block Size

| Block Size | Wavefronts per Block | Wasted Lanes per Block |
|:-----------|:---------------------|:-----------------------|
| 32 | 1 (half empty) | 32 |
| 64 | 1 | 0 |
| 96 | 2 (one half empty) | 32 |
| 128 | 2 | 0 |
| 256 | 4 | 0 |

### Exercise 5: Wavefront Analysis

**Question 1**: A kernel is launched with block size 100. How many wavefronts does each block require? How many lanes are wasted?

- Wavefronts per block: ______
- Wasted lanes: ______

**Question 2**: Why does AMD use a wavefront size of 64 instead of 32 like NVIDIA?

Your answer: ______

**Question 3**: What happens when threads within a wavefront take different branches (if-else)?

Your answer: ______

## 10. Occupancy

**Occupancy** is the ratio of active warps to the maximum warps a CU can support. Higher occupancy can help hide memory latency.

### Factors Limiting Occupancy

| Resource | How It Limits Occupancy |
|:---------|:------------------------|
| **Registers per thread** | More registers = fewer threads can fit on CU |
| **Shared memory per block** | More LDS = fewer blocks can fit on CU |
| **Block size** | Hardware limits max blocks per CU |

### Exercise 6: Occupancy Calculation

For MI100: Max 2048 threads per CU, Max 32 wavefronts per CU, Max 16 blocks per CU

| Block Size | Max Blocks (2048/block_size) | Limited by | Active Threads | Occupancy |
|:-----------|:-----------------------------|:-----------|:---------------|:----------|
| 64 | 32, but max 16 blocks | Block limit | 16 x 64 = ______ | ______% |
| 128 | ______ | ______ | ______ | ______% |
| 256 | ______ | ______ | ______ | ______% |
| 512 | ______ | ______ | ______ | ______% |

**Note**: Occupancy = Active Threads / 2048 x 100%

## 11. Experiment: Effect of Sub-Wavefront Block Sizes

What happens when block size is smaller than the wavefront size?

In [ ]:
%%bash
# Test with block size 32 (half warp on AMD)
echo "=== Block Size: 32 ==="
./vector_add_lab 1048576 32

In [ ]:
%%bash
# Test with block size 16
echo "=== Block Size: 16 ==="
./vector_add_lab 1048576 16

### Exercise 7: Observe the Impact

Compare the execution time of block size 16 vs 256:

- Block size 16 execution time: ______ ms
- Block size 256 execution time: ______ ms
- Ratio (16/256): ______

Explain why there is a difference:

_______________________________________________________________

_______________________________________________________________

## 12. Summary

### Key Takeaways

1. **Thread Hierarchy**: GPU threads are organized into Grids, Blocks, and Threads

2. **Hardware Mapping**: Blocks map to Compute Units; Warps (64 threads on AMD) execute on SIMDs

3. **Block Size Selection**:
   - Should be a multiple of warp size (64 on AMD, 32 on NVIDIA)
   - Common choices: 128, 256, or 512
   - Tradeoff between occupancy and register/shared memory pressure

4. **Thread Utilization**: Not all launched threads may do useful work when N is not divisible by block size

### Best Practices

| Guideline | Reason |
|:----------|:-------|
| Block size = multiple of warp size | Avoid partial warp waste |
| Block size between 128-512 | Balance occupancy and resource usage |
| Start with 256 threads per block | Good default for most kernels |

## 13. Challenge Exercise

Modify the kernel to process **multiple elements per thread**. This is a common technique when N is very large.

```cpp
__global__ void vector_add_multi(const float* A, const float* B, float* C, 
                                  int N, int elements_per_thread) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = _______________________________;  // Fill in: total number of threads
    
    for (int i = tid; i < N; i += stride) {
        C[i] = A[i] + B[i];
    }
}
```

**Question**: What is the advantage of this approach over the 1:1 thread-to-element mapping?

---

## Appendix: Reference Files

- `kernel.h` - Kernel function declaration
- `vector_add_kernel.hip` - Kernel implementation (student version)
- `vector_add_reference.hip` - Reference solution with annotations
- `main_lab.cpp` - Main program with timing and verification